# Implementing Binary Dasymetric Interpolation
## Following an NHGIS-like Methodology 

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

from tobler.area_weighted import area_interpolate
from tobler.dasymetric import extract_raster_features
from tobler.util import h3fy

%load_ext watermark
%watermark -a 'eli knaap' -v -d -u -p tobler,geopandas,rasterio

**Note**: This notebook relies on functionality from the [`contextily`](https://contextily.readthedocs.io/en/latest/) package that provides convenient basemaps for geospatial plots, the [`osmnx`](https://osmnx.readthedocs.io/en/stable/) package that provides access to OpenStreetMap data, and the [`pytidycensus`](https://pypi.org/project/pytidycensus/) package that provides an interface to the U.S. Census API. Set the `CENSUS_API_KEY` environment variable to your Census API key before running the data-fetching cells. Install the packages with  
  
`pip install contextily osmnx pytidycensus`     
or    
`conda install contextily osmnx -c conda-forge` followed by `pip install pytidycensus`

In [ ]:
import osmnx as ox

[NHGIS uses](https://www.nhgis.org/documentation/time-series/2000-blocks-to-2010-geog#advanced-model) a two step process, in which they first apply a binary dasymetric filter, then apply target density weighting. Here we will follow a similar strategy, relplicating their BD filtering (albeit with slightly different open data and replicable, open source tooling) before carrying out a simpler areal interpolation.

>In NHGIS's BD model for 2000 block data, the inhabited zone consists of all areas that are at least 5% developed impervious surface (within each 30-meter square cell of NLCD 2001 data) and lie within 300 feet of a residential road center line2 but not in a water body, using road definitions and water polygons from the 2010 TIGER/Line Shapefiles.

so we'll start by grabbing residential streets from OSM, buffering them out 300 feet, then using that geometry to select the impervious surfaces within

In [ ]:
rside_streets = ox.features_from_place(
    "Riverside County, California", tags={"highway": "residential"}
)
rside_streets = rside_streets.to_crs(2230)
rside_streets = rside_streets.set_geometry(rside_streets.buffer(300))

Now we use the street buffers to extract any cells between 5 and 100% impervious surface

First, download [NLCD](https://www.mrlc.gov/data) data from here: https://s3-us-west-2.amazonaws.com/mrlc/NLCD_2016_Impervious_L48_20190405.zip, then unzip the archive and pass the path to the `img` file to tobler's `extract_raster_features` function

In [ ]:
nlcd_path = "s3://spatial-ucr/nlcd/impervious/nlcd_impervious_2016.tif"

In [ ]:
import rasterio

with rasterio.Env(AWS_NO_SIGN_REQUEST="YES"):
    urban_rside = extract_raster_features(
        rside_streets, nlcd_path, pixel_values=list(range(5, 101)), collapse_values=True
    )

In [ ]:
urban_rside = urban_rside.to_crs(2230)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 14))
urban_rside.plot(ax=ax);

This geodataframe represents the areas that meet both criteria NHGIS suggests

Now we need some census data to transfer geometries. Let's grab tract-level median home values using pytidycensus. The cell below requires an API key, so we read the result from the cache. If you'd like to replicate this analysis for another area, use `pytidycensus` as indicated below.

In [ ]:
rside_census = gpd.read_file("cache/rside_census.gpkg")

And lets generate some synthetic zones to transfer this population data into using h3 hexagons

In [ ]:
rside_hex = h3fy(rside_census, resolution=8)
rside_hex.plot();

Ok, now all that's left is to clip our input census data, "restricting" it to the locations NLCD defines as 5-100% impervious. To do so, we just need to clip the census data using our extracted urban features as a mask

In [ ]:
dasy = rside_census.clip(urban_rside)

Clipping census geometries by the urban features leaves the census attributes intact but reshapes them. If a census tract had 2000 people in it and covered 1 sqmi, but only a quarter mile of the tract was urbanized, the new geodataframe effectively shows these 2000 people occupying the quarter mile. By passing these new data to the `areal_interpolate` function, we're still assuming that population density is constant across our new geometries (a condition that may not be true in reality) but that assumption is much more plausible than when using original census geometries. 

In [ ]:
dasy.plot();

In [ ]:
dasy.B25077_001E = dasy.B25077_001E.replace("<NA>", None).astype(float)

In [ ]:
dasy.B25077_001E

In [ ]:
interp = area_interpolate(dasy, rside_hex, extensive_variables=["B25077_001E"])

In [ ]:
interp["plot"] = np.log1p(interp.B25077_001E)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))
interp.to_crs(3857).plot("plot", ax=ax, alpha=0.4, tiles=True)
ax.axis("off");

And there we have it. Riverside's 2017 population estimated at a regular hexgrid